In [2]:
import pandas as pd
from pathlib import Path

In [3]:
data_path = Path("../data")
patient_files = list(data_path.rglob("*.psv"))

print("Number of patient files:", len(patient_files))

Number of patient files: 40336


In [4]:
df = pd.read_csv(patient_files[0], sep="|")

df.head()

,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,EtCO2,BaseExcess,HCO3,...,WBC,Fibrinogen,Platelets,Age,Gender,Unit1,Unit2,HospAdmTime,ICULOS,SepsisLabel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,73,1,1,0,-214.64,1,0
1,93.0,92.5,NaN,110.0,76.0,56.0,22.0,NaN,NaN,NaN,...,NaN,NaN,NaN,73,1,1,0,-214.64,2,0
2,91.0,96.0,NaN,108.0,84.5,72.0,23.5,NaN,NaN,NaN,...,NaN,NaN,NaN,73,1,1,0,-214.64,3,0
3,93.0,98.0,NaN,123.0,87.0,61.0,21.0,NaN,NaN,NaN,...,NaN,NaN,NaN,73,1,1,0,-214.64,4,0
4,93.0,95.0,NaN,110.0,81.0,70.0,20.0,NaN,NaN,NaN,...,NaN,NaN,NaN,73,1,1,0,-214.64,5,0


In [5]:
df.shape #shows the NUM OF OBSERVATIONS

(24, 41)

In [6]:
## NULL VALUES COUNTS

df.isna().sum().sort_values(ascending=False) 

EtCO2               24
BaseExcess          24
Alkalinephos        24
AST                 24
SaO2                24
PaCO2               24
pH                  24
FiO2                24
HCO3                24
Phosphate           24
Bilirubin_direct    24
Lactate             24
Bilirubin_total     24
PTT                 24
Fibrinogen          24
TroponinI           24
Chloride            24
Hgb                 23
Hct                 23
Calcium             23
Creatinine          23
WBC                 23
Platelets           23
BUN                 23
Potassium           22
Magnesium           22
Glucose             19
Temp                19
Resp                 2
O2Sat                2
HR                   2
SBP                  2
MAP                  2
DBP                  2
Age                  0
Gender               0
Unit1                0
Unit2                0
HospAdmTime          0
ICULOS               0
SepsisLabel          0
dtype: int64

In [7]:
##COUNT HOW MANY TIMES A PATIENT LABELLED SEPSIS OR NOT

df['SepsisLabel'].value_counts()

SepsisLabel
0    24
Name: count, dtype: int64

### Number of Patients with (1 or 0) SepsisLabel.
- I need to combine the data into 1 big csv file so it would be easier for analysis and further steps.
- Going further, the file used would be the combined data csv

In [8]:
sepsis_df = pd.read_csv("../data/processed/sepsis_combined.csv")

patient_labels = sepsis_df.groupby("PatientID")["SepsisLabel"].max() ##Either 1 or 0
counts = patient_labels.value_counts()

print(counts) 
    

SepsisLabel
0    37404
1     2932
Name: count, dtype: int64


In [24]:
sepsis_df.columns

Index(['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2',
       'BaseExcess', 'HCO3', 'FiO2', 'pH', 'PaCO2', 'SaO2', 'AST', 'BUN',
       'Alkalinephos', 'Calcium', 'Chloride', 'Creatinine', 'Bilirubin_direct',
       'Glucose', 'Lactate', 'Magnesium', 'Phosphate', 'Potassium',
       'Bilirubin_total', 'TroponinI', 'Hct', 'Hgb', 'PTT', 'WBC',
       'Fibrinogen', 'Platelets', 'Age', 'Gender', 'Unit1', 'Unit2',
       'HospAdmTime', 'ICULOS', 'SepsisLabel', 'PatientID'],
      dtype='str')

In [25]:
perc = (sepsis_df.isna().sum() / len(sepsis_df)) * 100
perc

HR                   9.882619
O2Sat               13.061119
Temp                66.162697
SBP                 14.576958
MAP                 12.451279
DBP                 31.345887
Resp                15.354559
EtCO2               96.286843
BaseExcess          94.579020
HCO3                95.810618
FiO2                91.665754
pH                  93.069688
PaCO2               94.440121
SaO2                96.549372
AST                 98.377604
BUN                 93.134434
Alkalinephos        98.393194
Calcium             94.116067
Chloride            95.460279
Creatinine          93.904433
Bilirubin_direct    99.807371
Glucose             82.894325
Lactate             97.329872
Magnesium           93.689578
Phosphate           95.986303
Potassium           90.689082
Bilirubin_total     98.509158
TroponinI           99.047745
Hct                 91.145979
Hgb                 92.617558
PTT                 97.055875
WBC                 93.593199
Fibrinogen          99.340167
Platelets 

- Divide the sepsis_df into smaller dataframes.
- Dataframes are classified as: Vital signs (8), Biomarker(26), Demographics (6)

In [26]:
vital_signs = ["HR", "O2Sat", "Temp", "SBP", "MAP", "DBP", "Resp", "EtCO2", "SepsisLabel", "PatientID"]

lab_features = [
    "BaseExcess", "HCO3", "FiO2", "pH", "PaCO2",
    "SaO2", "AST", "BUN", "Alkalinephos", "Calcium",
    "Chloride", "Creatinine", "Bilirubin_direct",
    "Glucose", "Lactate", "Magnesium", "Phosphate",
    "Potassium", "Bilirubin_total", "TroponinI",
    "Hct", "Hgb", "PTT", "WBC", "Fibrinogen", "Platelets", "SepsisLabel", "PatientID"
]

demographics = [
    "Age", "Gender", "Unit1", "Unit2",
    "HospAdmTime", "ICULOS", "SepsisLabel", "PatientID"
]

- Average missing percentage across the 8 vital signs.

In [32]:
vital_df = sepsis_df[vital_signs]

vital_missing = vital_df.isna().sum() / len(vital_df) * 100
print("Avg missing vital perct:", vital_missing.mean())

Avg missing vital perct: 25.912196159024877


In [33]:
lab_features_df = sepsis_df[lab_features]

lab_missing = lab_features_df.isna().sum() / len(lab_features_df) * 100
print("Avg missing lab perct:", lab_missing.mean())

Avg missing lab perct: 88.04522491894045


In [34]:
demo_df = sepsis_df[demographics]

demo_missing = demo_df.isna().sum() / len(demo_df) * 100
print("Avg missing demo perct:", demo_missing.mean())

Avg missing demo perct: 9.856333872349747
